# Competencia — modelado reproducible v2

Este notebook reproduce el candidato congelado de 238 features. El EDA y los experimentos históricos permanecen en `competencia.ipynb`.

## 1. Configuración e imports

In [ ]:
from pathlib import Path
import gc
import os
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-codex')

import numpy as np
import pandas as pd
from IPython.display import display

PROYECTO = Path.cwd().resolve()
if not (PROYECTO / 'preprocesamiento.py').is_file():
    raise FileNotFoundError(
        'Ejecute este notebook con el directorio del proyecto como cwd.'
    )
sys.path.insert(0, str(PROYECTO))

import feature_engineering
from preprocesamiento import (
    auditar_features,
    auditar_preprocesamiento,
    cargar_competencia_01,
    cargar_lista_features,
    convertir_infinitos_en_nulos,
    preprocesar_competencia_01,
    resolver_ruta_fuente,
    resumir_nulos_por_familia,
    seleccionar_entrada_feature_engineering,
)
from modelo_referencia import (
    MES_VALIDACION_REFERENCIA,
    MESES_ENTRENAMIENTO_REFERENCIA,
    cargar_y_validar_configuracion,
    comparar_con_validacion_historica,
    dividir_temporalmente,
    entrenar_y_evaluar,
    guardar_resultados,
)

RUTA_FUENTE_EXPLICITA = None  # o Path('/ruta/competencia_01.csv')
RUTA_FUENTE = resolver_ruta_fuente(
    RUTA_FUENTE_EXPLICITA, directorio_proyecto=PROYECTO
)
RUTA_FEATURES = PROYECTO / 'features_candidato_fe.csv'
RUTA_CONFIGURACION = PROYECTO / 'configuracion_modelo_final.json'
RUTA_VALIDACION_HISTORICA = PROYECTO / 'validacion_ensemble_trial25.csv'
DIRECTORIO_RESULTADOS = PROYECTO / 'resultados_modelado_v2'

PARTICION = {
    'meses_train': MESES_ENTRENAMIENTO_REFERENCIA,
    'mes_validacion': MES_VALIDACION_REFERENCIA,
}
EJECUTAR_MODELOS = True
GUARDAR_MODELOS = False
TOLERANCIA_GANANCIA_REFERENCIA = 5_500_000
TOLERANCIA_AUC_REFERENCIA = 0.003

print('Fuente:', RUTA_FUENTE)
print('Partición:', PARTICION)

## 2. Carga de la fuente

In [ ]:
base_original = cargar_competencia_01(RUTA_FUENTE)
print(f'Filas: {len(base_original):,}')
print(f'Columnas originales: {base_original.shape[1]}')
print('Meses:', sorted(base_original['foto_mes'].unique()))

## 3. Preprocesamiento

In [ ]:
base_preprocesada, reporte_correcciones = preprocesar_competencia_01(
    base_original
)
del base_original
gc.collect()
display(reporte_correcciones)
print('Las 13 features de anclaje y target_baja2 fueron reconstruidos.')

## 4. Auditoría de calidad

In [ ]:
auditoria_preprocesamiento = auditar_preprocesamiento(base_preprocesada)
display(auditoria_preprocesamiento)
print('AUDITORÍA DE CALIDAD: OK')

## 5. Feature engineering

Se reutiliza `construir_features_candidato` sin agregar los bloques descartados. Las seis fotos permanecen juntas hasta terminar los lags; mayo aporta historia a junio aunque no participe del entrenamiento.

In [ ]:
features_finales = cargar_lista_features(RUTA_FEATURES)
entrada_fe, features_entrada = seleccionar_entrada_feature_engineering(
    base_preprocesada, features_finales
)
entrada_fe, infinitos_entrada_convertidos = convertir_infinitos_en_nulos(
    entrada_fe, features_entrada, copiar=False
)
del base_preprocesada
gc.collect()

base_modelado, features_lag, features_carga = (
    feature_engineering.construir_features_candidato(entrada_fe)
)
del entrada_fe
gc.collect()

print('Features de entrada:', len(features_entrada))
print('Lag/delta/eventos:', len(features_lag))
print('Carga financiera:', len(features_carga))
print('Features finales:', len(features_finales))
print('Infinitos de entrada convertidos a nulo:', infinitos_entrada_convertidos)

## 6. Auditoría de features

In [ ]:
auditoria_features = auditar_features(
    base_modelado,
    features_finales,
    features_lag,
    features_carga,
    feature_engineering.VARIABLES_LAG1,
)
nulos_por_familia = resumir_nulos_por_familia(
    base_modelado, features_finales, features_lag, features_carga
)
display(pd.DataFrame([auditoria_features]))
display(nulos_por_familia)
print('AUDITORÍA DE FEATURES: OK')

## 7. División temporal

In [ ]:
particion = dividir_temporalmente(
    base_modelado,
    features_finales,
    meses_train=PARTICION['meses_train'],
    mes_validacion=PARTICION['mes_validacion'],
)
print('Train:', particion.meses_train, f'({len(particion.X_train):,} filas)')
print('Validación:', particion.mes_validacion, f'({len(particion.X_valid):,} filas)')
print('Mayo queda fuera del fit y permanece sólo como historia para junio.')

## 8. Modelo de referencia

In [ ]:
configuracion = cargar_y_validar_configuracion(RUTA_CONFIGURACION)
if not EJECUTAR_MODELOS:
    raise RuntimeError('EJECUTAR_MODELOS=False: no se generaron resultados.')

directorio_modelos = (
    DIRECTORIO_RESULTADOS / 'modelos' if GUARDAR_MODELOS else None
)
resultado = entrenar_y_evaluar(
    particion, configuracion, directorio_modelos=directorio_modelos
)
display(resultado.metricas_semillas)

## 9. Evaluación del ensemble

In [ ]:
comparacion_historica = None
if RUTA_VALIDACION_HISTORICA.is_file():
    comparacion_historica = comparar_con_validacion_historica(
        resultado.predicciones_validacion, RUTA_VALIDACION_HISTORICA
    )

ruta_resultados = guardar_resultados(
    resultado,
    DIRECTORIO_RESULTADOS,
    reporte_correcciones=reporte_correcciones,
    auditoria_preprocesamiento=auditoria_preprocesamiento,
    auditoria_features=auditoria_features,
    nulos_por_familia=nulos_por_familia,
    comparacion_historica=comparacion_historica,
)

display(resultado.metricas_ensemble)
display(resultado.correlacion_semillas.round(4))
if comparacion_historica is not None:
    display(comparacion_historica)

diferencia_importante = (
    resultado.metricas_ensemble['delta_ganancia'].abs()
    .gt(TOLERANCIA_GANANCIA_REFERENCIA)
    | resultado.metricas_ensemble['delta_auc'].abs()
    .gt(TOLERANCIA_AUC_REFERENCIA)
)
if diferencia_importante.any():
    raise RuntimeError(
        'La reproducción difiere materialmente de la referencia; '        'revise versiones, orden de datos y configuración.'
    )
print('Resultados guardados en:', ruta_resultados)
print('REPRODUCCIÓN DEL CANDIDATO: OK')

## 10. Punto de partida para nuevos experimentos

La reproducción queda congelada. Los próximos experimentos deben partir de `base_modelado`, mantener las auditorías y cambiar explícitamente una sola decisión por vez. La partición se controla desde `PARTICION`; no se incorporaron nuevos lags, undersampling, targets alternativos, segmentación, ensembles ni búsquedas de Optuna.